In [ ]:
%pip install langchain
%pip install langchain-community
%pip install langchain-core
%pip install langchain-openai
%pip install langchain-pinecone
%pip install pinecone
%pip install pinecone-client
%pip install pypdf
%pip install python-dotenv
%pip install beautifulsoup4
%pip install huggingface-hub
%pip install -U "langchain[huggingface]"
%pip install -qU sentence_transformers


In [11]:
from pinecone import Pinecone, ServerlessSpec
import os

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

pc.create_index(
    name="jurisdesk",
    dimension=384,          # bge-small-en = 384
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)


{
    "name": "jurisdesk",
    "metric": "cosine",
    "host": "jurisdesk-42d5750.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [ ]:

from bs4 import BeautifulSoup
import requests
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re
from langchain_core.documents import Document
from pinecone import Pinecone
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from dotenv import load_dotenv
load_dotenv()

FirstLink = "https://www.indiacode.nic.in/handle/123456789/1362/browse?type=shorttitle&rpp=845"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.indiacode.nic.in/"
}

session = requests.Session()
html = session.get(FirstLink, headers=headers).text

soup = BeautifulSoup(html)

# print(soup.prettify())



def universal_legal_splitter(text, max_len=1200, overlap=200):
    chunks = []
    starts = [0]

    # find anchor positions
    for pattern in ANCHORS:
        for m in re.finditer(pattern, text):
            starts.append(m.start())

    starts = sorted(set(starts))
    starts.append(len(text))

    # build chunks from anchors
    for i in range(len(starts) - 1):
        chunk = text[starts[i]:starts[i+1]].strip()
        if chunk:
            chunks.append(chunk)

    # now enforce size limits (lossless)
    final_chunks = []
    for chunk in chunks:
        if len(chunk) <= max_len:
            final_chunks.append(chunk)
        else:
            for i in range(0, len(chunk), max_len - overlap):
                final_chunks.append(chunk[i:i+max_len])

    return final_chunks

def clean_text(text):
    # footnote markers like 1[, 2[
    text = re.sub(r"\n?\d+\[", "\n", text)

    # page numbers alone on a line
    text = re.sub(r"\n\d+\n", "\n", text)

    return text


ANCHORS = [
    r"\nCHAPTER\s+[IVXLC]+[A-Z]*",
    r"\n\d+[A-Z]?\.\s",          # Sections
    r"\nArticle\s+\d+",          # Articles
]

FAILED_LOG = "failed_pdfs.txt"


# ---- embeddings ----
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# ---- pinecone client ----
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index = pc.Index("jurisdesk")  # must exist

# ---- vector store ----
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace="state_acts",
)



firstAs= soup.select("table tr td a")

baseUrl= "https://www.indiacode.nic.in"
seen_urls = set()
pdf_counter = 277
for i in range(278,len(firstAs)):
# for a in firstAs:
    a= firstAs[i]
    try:
        href = a.get("href")
        if not href:
            continue

        fullUrl = baseUrl + href
        session = requests.Session()
        html = session.get(fullUrl, headers=headers).text

        soup2 = BeautifulSoup(html)
        aa = soup2.select_one("a[href$='.pdf']")
        if not aa:
            raise ValueError("PDF link not found")

        pdf_url = baseUrl + aa.get("href")

        if pdf_url in seen_urls:
            continue
        seen_urls.add(pdf_url)

        pdf_counter += 1
        print(f"Processing PDF #{pdf_counter}")

        # ---- load PDF ----
        loader = PyPDFLoader(pdf_url)
        docs = loader.load()

        docs = [
            d for d in docs
            if "ARRANGEMENT OF SECTIONS" not in d.page_content.upper()
        ]

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1200,
            chunk_overlap=200,
            separators=["\n\n", "\n", ". ", " ", ""]
        )

        splits = splitter.split_documents(docs)

        enriched_docs = [
            Document(
                page_content=d.page_content,
                metadata={
                    **d.metadata,
                    "doc_type": "state_act",
                    "source": "indiacode",
                    "url": pdf_url,
                    "pdf_number": pdf_counter
                }
            )
            for d in splits
        ]

        vectorstore.add_documents(enriched_docs)

    except Exception as e:
        with open(FAILED_LOG, "a") as f:
            f.write(
                f"PDF #{pdf_counter} | URL: {pdf_url if 'pdf_url' in locals() else 'UNKNOWN'} | ERROR: {str(e)}\n"
            )
        print(f" Failed PDF #{pdf_counter}")
        continue



    

Processing PDF #278
Processing PDF #279
Processing PDF #280
Processing PDF #281
Processing PDF #282
 Failed PDF #282
Processing PDF #283
Processing PDF #284
Processing PDF #285
Processing PDF #286
Processing PDF #287
Processing PDF #288
Processing PDF #289
Processing PDF #290
Processing PDF #291
Processing PDF #292
Processing PDF #293
Processing PDF #294
Processing PDF #295
Processing PDF #296
Processing PDF #297
Processing PDF #298
Processing PDF #299
Processing PDF #300
Processing PDF #301
Processing PDF #302
Processing PDF #303
Processing PDF #304
Processing PDF #305
Processing PDF #306
Processing PDF #307
Processing PDF #308
Processing PDF #309
 Failed PDF #309
Processing PDF #310
Processing PDF #311
Processing PDF #312
Processing PDF #313
Processing PDF #314
Processing PDF #315
Processing PDF #316
Processing PDF #317
Processing PDF #318
Processing PDF #319
Processing PDF #320
Processing PDF #321
Processing PDF #322
Processing PDF #323
Processing PDF #324
Processing PDF #325
Proces